# APMC catalog -- page boundary and Images Above correction

The OCR'd CSV at `wip/in/<BASENAME>.csv` got some entries assigned to the wrong
catalog page because the upstream page-split confused the OCR, and the
`Images Above` column has known mis-counts on entries the OCR claims have
at least one stamp image above them. This notebook uses the Claude vision
API to (a) verify the first and last entries of each page against the
matching half-page PNG and slide entries between adjacent pages until both
ends agree, and (b) re-count stamp images above each entry the OCR
already claims has `Images Above > 0`, auto-correcting where the model
disagrees. Output goes to `wip/out/<BASENAME>.csv`.

Stages:

1. Setup and config.
2. Load CSV.
3. Build the claimed-endpoints table.
4. Extract observed endpoints from PNGs via vision API (cached to disk).
5. Compare claimed vs observed, flag mismatches.
6. Manual override (optional).
7. Apply page-slide corrections.
8. Write corrected CSV (intermediate; section 14 overwrites it).
9. Page-correction verification notes.
10. Images Above verification overview.
11. System prompt and per-page vision call (counts only).
12. Cache + loop: one API call per page, restricted to non-zero rows.
13. Flag deltas and apply Images Above corrections.
14. Manual override + final CSV write.


In [1]:
# 1. Setup and config
import base64, json, os, re
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import anthropic

# Repo-root .env (notebook cwd is tools/).
load_dotenv(Path("..") / ".env")

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

BASENAME = "VA_ASCC_CTLG"
INPUT_CSV  = Path(f"./wip/in/{BASENAME}.csv")
IMAGES_DIR = Path(f"./wip/in/{BASENAME}")
OUTPUT_CSV = Path(f"./wip/out/{BASENAME}.csv")
CACHE_FILE = Path(f"./wip/cache/{BASENAME}_vision.json")
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

VISION_MODEL    = "claude-sonnet-4-6"
PROMPT_VERSION  = "v1"
# Two thresholds: permissive for surfacing mismatches, strict for accepting an
# anchor and moving rows. Both apply to NORMALIZED text (see section 5).
FLAG_THRESHOLD  = 0.6   # below this in section 5, page is flagged needs_fix
SLIDE_THRESHOLD = 0.85  # below this in section 7, anchor is rejected
FORCE_REFRESH   = False

assert INPUT_CSV.exists(), f"missing input CSV: {INPUT_CSV}"
assert IMAGES_DIR.is_dir(), f"missing images dir: {IMAGES_DIR}"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env (check repo-root .env)"

client = anthropic.Anthropic()

print(f"basename:    {BASENAME}")
print(f"input csv:   {INPUT_CSV}")
print(f"images dir:  {IMAGES_DIR}")
print(f"output csv:  {OUTPUT_CSV}")
print(f"cache file:  {CACHE_FILE}")
print(f"model:       {VISION_MODEL}  (prompt {PROMPT_VERSION})")


basename:    VA_ASCC_CTLG
input csv:   wip/in/VA_ASCC_CTLG.csv
images dir:  wip/in/VA_ASCC_CTLG
output csv:  wip/out/VA_ASCC_CTLG.csv
cache file:  wip/cache/VA_ASCC_CTLG_vision.json
model:       claude-sonnet-4-6  (prompt v1)


In [2]:
# 2. Load CSV
df = pd.read_csv(INPUT_CSV, dtype={"Page": int, "Images Above": int}).reset_index(drop=True)
df["_orig_row"] = df.index
print(f"rows: {len(df):,};  pages: {df.Page.min()}-{df.Page.max()}")
print()
print(df.groupby("Page").size().rename("rows").to_frame())
df.head(3)


rows: 1,442;  pages: 419-435

      rows
Page      
419     36
420     45
421     42
422     60
423     67
424     68
425     60
426     56
427     68
428     47
429     83
430     76
431     48
432    119
433    160
434    159
435    248


,Listing,Page,Images Above,Default Shape,Manuscript,Postmark Key,State Code,State Name,_orig_row
0,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00",419,1,NaN,NaN,APMC-VA-1,VA,Virginia,0
1,"(L)(Sept.15,1774) . . .. 1000.00",419,0,NaN,NaN,APMC-VA-2,VA,Virginia,1
2,"Colchester(June 30,1774;Ms;Black) .. 1500.00",419,0,NaN,NaN,APMC-VA-3,VA,Virginia,2


In [3]:
# 3. Build the claimed-endpoints table.
# The CSV is in catalog reading order, so per page the first and last rows are
# the claimed first and last entries on that page.

def claimed_endpoints(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for page, sub in df.groupby("Page", sort=True):
        rows.append({
            "page": int(page),
            "claimed_first_idx":  int(sub.index[0]),
            "claimed_first_text": sub.iloc[0]["Listing"],
            "claimed_last_idx":   int(sub.index[-1]),
            "claimed_last_text":  sub.iloc[-1]["Listing"],
            "row_count":          len(sub),
        })
    return pd.DataFrame(rows).set_index("page")

claimed = claimed_endpoints(df)
claimed


,claimed_first_idx,claimed_first_text,claimed_last_idx,claimed_last_text,row_count
page,,,,,
419,0,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00",35,(L)(Aug.1776) . .. 500.00,36
420,36,"Alexa.(E)(Sept.17,1776;Ms;Black) . 500.00",80,"Same(July 16,1778;Ms;Magenta) .... 600.00",45
421,81,"WILLIAMS'B.G.(E)(July 2,1788;SL-34x2.5,MDD;PAID;Black) . . . 750.00",122,BATTLETOWN/VA.(1830;28;Black) ........ . 50.00,42
422,123,BELMONT/Va.(1839;-;Black) ................. 50.00,182,"Same(1833;29,NOR;Red,Black) . . . . . . . . . . . ...... 40.00",60
423,183,"(1)CLARKSBURG/Va.(""a""high)(1835-47;30;PAID;Black) . 30.00",249,"FREDG.VA(""G""&""A""high)(1800-25;26;FREE;Red,Black) 60.00",67
424,250,"FREDG VA(""G""&""A""high,small curved fleuron)(1818-28;30;FREE;Red,Black) . . . . . . . . . . . . . . . . . . . . . . . . 50.00",317,KEMPSVILLE/Va.(1850's;30;Black) ................... 40.00,68
425,318,"KEYSVILLE/VA(1855;30;PAID,3,PAID/3[C];Black) ....... 30.00",377,MARYSVILLE/Va.(1853;-;Green) . . . . . . . . . . . . . . . . . 40.00,60
426,378,"MIDDLH.VA.(""H""&""A""high)(1835-39;31;PAID;Black) ...... 35.00",433,"Norfolk(1789-90;SL-31x4.5,MDD;Black) ......... 200.00",56
427,434,"NORFOLK(1789-99;SL-31x4.5,MD;Black) . . ........ 150.00",501,PORT ROYAL VA.(1833;SL-33x3.5;Black) .... 200.00,68


## 4. Vision extraction

Send each `page-NNNN-L.png` and `page-NNNN-R.png` to Claude vision, asking for
the topmost (`-L`) or bottommost (`-R`) entry text as strict JSON. Responses
are cached to disk; subsequent runs read the cache and make zero API calls.
Set `FORCE_REFRESH=True` in the config cell to re-query.

The system prompt is sent with `cache_control: ephemeral` so subsequent calls
within the 5-minute TTL pay the cached rate on system tokens.


In [4]:
# 4a. System prompt for the vision model. Frozen string so prompt caching hits.
SYSTEM_PROMPT = """You are reading half-page scans of an old American Stampless Cover (ASCC) catalog.

Each catalog page is split into two halves: -L (left) and -R (right). Each
half-page contains a column of catalog entries, one entry per line. An entry
typically looks like a place name followed by parenthetical metadata (date,
manuscript-or-printed indicator, ink color), then a row of dots, then a price
at the end. Examples drawn verbatim from the CSV:

  Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00
  Fredg.(Aug.2,1772;Ms;Black) .. 1500.00
  (L)(Sept.15,1774) . . .. 1000.00

Some entries are continuation lines that start with an open paren -- those are
still entries. Page numbers, section headers, and illustrations above the
entries are NOT entries; skip them.

When asked for the FIRST entry, return the topmost entry visible on the
half-page. When asked for the LAST entry, return the bottommost entry visible
on the half-page. Return the entry text as you literally see it in the scan.

Return STRICT JSON only -- no surrounding prose, no markdown fences. Two valid
shapes:

  {"first_entry_text": "Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00"}
  {"last_entry_text":  "(L)(Aug.1776) . .. 500.00"}
"""


In [5]:
# 4b. Vision call. Returns parsed JSON dict from the response.

def extract_boundary_entry(image_path: Path, position: str) -> dict:
    assert position in ("first", "last")
    key = "first_entry_text" if position == "first" else "last_entry_text"
    half = "top half (-L)" if position == "first" else "bottom half (-R)"
    user_prompt = (
        f"This image is the {half} of one ASCC catalog page. "
        f"Identify the {position.upper()} entry visible on this half-page "
        f'and return strict JSON of the form {{"{key}": "..."}}.'
    )
    msg = client.messages.create(
        model=VISION_MODEL,
        max_tokens=1024,
        system=[{
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": base64.standard_b64encode(image_path.read_bytes()).decode(),
            }},
            {"type": "text", "text": user_prompt},
        ]}],
    )
    text = next(b.text for b in msg.content if b.type == "text").strip()
    # Tolerate a stray markdown fence even though the system prompt forbids it.
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.S)
    return json.loads(text)


In [6]:
# 4c. Cache + loop. Disk cache is invalidated when model or prompt_version changes.

def load_cache(path: Path) -> dict:
    if not path.exists():
        return {"model": VISION_MODEL, "prompt_version": PROMPT_VERSION, "responses": {}}
    cache = json.loads(path.read_text())
    if cache.get("model") != VISION_MODEL or cache.get("prompt_version") != PROMPT_VERSION:
        print(f"cache invalidated (was model={cache.get('model')!r}, prompt={cache.get('prompt_version')!r})")
        return {"model": VISION_MODEL, "prompt_version": PROMPT_VERSION, "responses": {}}
    return cache

def save_cache(path: Path, cache: dict) -> None:
    path.write_text(json.dumps(cache, indent=2))

cache = load_cache(CACHE_FILE)
responses = cache["responses"]

calls_made = 0
for page in claimed.index:
    for half, position in [("L", "first"), ("R", "last")]:
        key = f"page-{page:04d}-{half}"
        if key in responses and not FORCE_REFRESH:
            continue
        img_path = IMAGES_DIR / f"{key}.png"
        if not img_path.exists():
            print(f"missing image, skipping: {img_path}")
            continue
        try:
            result = extract_boundary_entry(img_path, position)
        except Exception as exc:
            print(f"  {key}: error {exc!r}")
            continue
        responses[key] = result
        save_cache(CACHE_FILE, cache)
        calls_made += 1
        snippet = list(result.values())[0]
        print(f"  {key}: {snippet[:80]!r}")

print(f"\napi calls this run: {calls_made};  cache size: {len(responses)}")

observed_rows = []
for page in claimed.index:
    L = responses.get(f"page-{page:04d}-L", {})
    R = responses.get(f"page-{page:04d}-R", {})
    observed_rows.append({
        "page": int(page),
        "observed_first_text": L.get("first_entry_text"),
        "observed_last_text":  R.get("last_entry_text"),
    })
observed = pd.DataFrame(observed_rows).set_index("page")
boundaries = claimed.join(observed)
boundaries



api calls this run: 0;  cache size: 34


,claimed_first_idx,claimed_first_text,claimed_last_idx,claimed_last_text,row_count,observed_first_text,observed_last_text
page,,,,,,,
419,0,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00",35,(L)(Aug.1776) . .. 500.00,36,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00","(L)(July 9,1777) ................................ 500.00"
420,36,"Alexa.(E)(Sept.17,1776;Ms;Black) . 500.00",80,"Same(July 16,1778;Ms;Magenta) .... 600.00",45,"ALEX(E)(Jan.14,1788;SL-14x3,MDD;Black) . . . . . . . . . . 400.00","Same(July 16,1778;Ms;Magenta) . . . . . . . . . . . . . . . . . . . . 600.00"
421,81,"WILLIAMS'B.G.(E)(July 2,1788;SL-34x2.5,MDD;PAID;Black) . . . 750.00",122,BATTLETOWN/VA.(1830;28;Black) ........ . 50.00,42,"WILLIAMS'B.G.(E)(July 2,1788;SL-34x2.5,MDD;PAID;Black) ......... 750.00",BATTLETOWN/VA.(1830;28;Black) . . . . . . . . . . . . . . . . 50.00
422,123,BELMONT/Va.(1839;-;Black) ................. 50.00,182,"Same(1833;29,NOR;Red,Black) . . . . . . . . . . . ...... 40.00",60,BELMONT/Va.(1839;--;Black) . . . . . . . . . . . . . . . . . . . . . . . 50.00,"Same(1833;29,NOR;Red,Black) ..................... 40.00"
423,183,"(1)CLARKSBURG/Va.(""a""high)(1835-47;30;PAID;Black) . 30.00",249,"FREDG.VA(""G""&""A""high)(1800-25;26;FREE;Red,Black) 60.00",67,"(1)CLARKSBURG/Va.(""a""high)(1835-47;30;PAID;Black) . . . 30.00","FREDG.VA.(""G""&""A""high)(1800-25;26;FREE;Red,Black) ... 60.00"
424,250,"FREDG VA(""G""&""A""high,small curved fleuron)(1818-28;30;FREE;Red,Black) . . . . . . . . . . . . . . . . . . . . . . . . 50.00",317,KEMPSVILLE/Va.(1850's;30;Black) ................... 40.00,68,"FREDG VA(""G""&""A""high,small curved fleuron)(1818-28;30;FREE;Red,Black) . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 50.00",KEMPSVILLE/Va.(1850's;30;Black) . . . . . . . . . . . . . . . . . . . . 40.00
425,318,"KEYSVILLE/VA(1855;30;PAID,3,PAID/3[C];Black) ....... 30.00",377,MARYSVILLE/Va.(1853;-;Green) . . . . . . . . . . . . . . . . . 40.00,60,"KEYSVILLE/VA.(1855;30;PAID,3,PAID/3[C];Black) ........ 30.00",MARYSVILLE/Va.(1853;--;Green) ........... 40.00
426,378,"MIDDLH.VA.(""H""&""A""high)(1835-39;31;PAID;Black) ...... 35.00",433,"Norfolk(1789-90;SL-31x4.5,MDD;Black) ......... 200.00",56,"MIDDLH.VA.(""H""&""A""high)(1835-39;31;PAID;Black) ...... 35.00","Norfolk(1789-90;SL-31x4.5,MDD;Black) ........... 200.00"
427,434,"NORFOLK(1789-99;SL-31x4.5,MD;Black) . . ........ 150.00",501,PORT ROYAL VA.(1833;SL-33x3.5;Black) .... 200.00,68,NORFOLK(1789-99;SL-31x4.5;MD;Black) . . . . . . . . . . . . 150.00,PORT ROYAL VA.(1833;SL-33x3.5;Black) ........... 200.00


## 5. Compare claimed vs observed

We normalize listings before fuzzy matching: lowercase, collapse runs of dots
and whitespace into a single space. Vision text is dot/space heavy
(`Mint Spring . . . . . . . . . . . . 1850 . . . . 20.00`); CSV text is not
(`Mint Spring . 1850 . 20.00`). Without normalization the place-name signal
gets drowned by dot/space noise and unrelated rows score as high as the true
match.

Rows where either end's normalized ratio falls below `FLAG_THRESHOLD` get
`needs_fix=True` and highlighted. (The stricter `SLIDE_THRESHOLD` only gates
acceptance of an anchor in section 7; this section is for visual review.)


In [7]:
# 5. Comparison with normalized fuzzy match.

def normalize(s) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    # Collapse whitespace and dots into a single space. This is the key fix:
    # dot/space noise dominates SequenceMatcher.ratio on these listings.
    s = re.sub(r"[\s\.]+", " ", s).strip()
    return s

def ratio(a, b) -> float:
    """Normalized fuzzy score with asymmetric containment.

    Vision sometimes returns a short form of a listing whose CSV form is
    longer (e.g. vision reads 'or Salem ... 40.00' for an entry whose CSV
    listing is 'Salem Bot(etourt Co) or Salem . 1819-44 ..... 40.00').
    The standard SequenceMatcher.ratio is symmetric and penalizes the
    length disparity even when the shorter string is fully contained in
    the longer. We take the max of the symmetric ratio and the
    asymmetric containment (matches / min(len)) so that 'shorter fully
    in longer' scores 1.0.
    """
    if a is None or b is None:
        return 0.0
    na, nb = normalize(a), normalize(b)
    if not na or not nb:
        return 0.0
    sm = SequenceMatcher(None, na, nb)
    matches = sum(blk.size for blk in sm.get_matching_blocks())
    symmetric  = 2 * matches / (len(na) + len(nb))
    asymmetric = matches / min(len(na), len(nb))
    return max(symmetric, asymmetric)

def compute_match(boundaries: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = boundaries.copy()
    out["first_ratio"] = [ratio(c, o) for c, o in zip(out["claimed_first_text"], out["observed_first_text"])]
    out["last_ratio"]  = [ratio(c, o) for c, o in zip(out["claimed_last_text"],  out["observed_last_text"])]
    out["first_match"] = out["first_ratio"] >= threshold
    out["last_match"]  = out["last_ratio"]  >= threshold
    out["needs_fix"]   = ~(out["first_match"] & out["last_match"])
    return out

boundaries = compute_match(boundaries, FLAG_THRESHOLD)
view_cols = ["claimed_first_text", "observed_first_text", "first_ratio",
             "claimed_last_text",  "observed_last_text",  "last_ratio",
             "needs_fix"]

def highlight(row):
    return ["background-color: #ffe0e0" if row["needs_fix"] else "" for _ in row]

boundaries[view_cols].style.apply(highlight, axis=1)


,claimed_first_text,observed_first_text,first_ratio,claimed_last_text,observed_last_text,last_ratio,needs_fix
page,,,,,,,
419,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00","Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00",1.000000,(L)(Aug.1776) . .. 500.00,"(L)(July 9,1777) ................................ 500.00",0.850000,False
420,"Alexa.(E)(Sept.17,1776;Ms;Black) . 500.00","ALEX(E)(Jan.14,1788;SL-14x3,MDD;Black) . . . . . . . . . . 400.00",0.717949,"Same(July 16,1778;Ms;Magenta) .... 600.00","Same(July 16,1778;Ms;Magenta) . . . . . . . . . . . . . . . . . . . . 600.00",1.000000,False
421,"WILLIAMS'B.G.(E)(July 2,1788;SL-34x2.5,MDD;PAID;Black) . . . 750.00","WILLIAMS'B.G.(E)(July 2,1788;SL-34x2.5,MDD;PAID;Black) ......... 750.00",1.000000,BATTLETOWN/VA.(1830;28;Black) ........ . 50.00,BATTLETOWN/VA.(1830;28;Black) . . . . . . . . . . . . . . . . 50.00,1.000000,False
422,BELMONT/Va.(1839;-;Black) ................. 50.00,BELMONT/Va.(1839;--;Black) . . . . . . . . . . . . . . . . . . . . . . . 50.00,1.000000,"Same(1833;29,NOR;Red,Black) . . . . . . . . . . . ...... 40.00","Same(1833;29,NOR;Red,Black) ..................... 40.00",1.000000,False
423,"(1)CLARKSBURG/Va.(""a""high)(1835-47;30;PAID;Black) . 30.00","(1)CLARKSBURG/Va.(""a""high)(1835-47;30;PAID;Black) . . . 30.00",1.000000,"FREDG.VA(""G""&""A""high)(1800-25;26;FREE;Red,Black) 60.00","FREDG.VA.(""G""&""A""high)(1800-25;26;FREE;Red,Black) ... 60.00",1.000000,False
424,"FREDG VA(""G""&""A""high,small curved fleuron)(1818-28;30;FREE;Red,Black) . . . . . . . . . . . . . . . . . . . . . . . . 50.00","FREDG VA(""G""&""A""high,small curved fleuron)(1818-28;30;FREE;Red,Black) . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 50.00",1.000000,KEMPSVILLE/Va.(1850's;30;Black) ................... 40.00,KEMPSVILLE/Va.(1850's;30;Black) . . . . . . . . . . . . . . . . . . . . 40.00,1.000000,False
425,"KEYSVILLE/VA(1855;30;PAID,3,PAID/3[C];Black) ....... 30.00","KEYSVILLE/VA.(1855;30;PAID,3,PAID/3[C];Black) ........ 30.00",1.000000,MARYSVILLE/Va.(1853;-;Green) . . . . . . . . . . . . . . . . . 40.00,MARYSVILLE/Va.(1853;--;Green) ........... 40.00,1.000000,False
426,"MIDDLH.VA.(""H""&""A""high)(1835-39;31;PAID;Black) ...... 35.00","MIDDLH.VA.(""H""&""A""high)(1835-39;31;PAID;Black) ...... 35.00",1.000000,"Norfolk(1789-90;SL-31x4.5,MDD;Black) ......... 200.00","Norfolk(1789-90;SL-31x4.5,MDD;Black) ........... 200.00",1.000000,False
427,"NORFOLK(1789-99;SL-31x4.5,MD;Black) . . ........ 150.00",NORFOLK(1789-99;SL-31x4.5;MD;Black) . . . . . . . . . . . . 150.00,0.976190,PORT ROYAL VA.(1833;SL-33x3.5;Black) .... 200.00,PORT ROYAL VA.(1833;SL-33x3.5;Black) ........... 200.00,1.000000,False


## 6. Manual override (optional)

If a row above is flagged `needs_fix=True` but you can visually confirm the
vision call was wrong (open the matching PNG to check), edit `boundaries`
directly in the cell below, then re-run this cell. Example:

    boundaries.at[420, "observed_first_text"] = "(L)(Dec.1779) ... 750.00"
    boundaries = compute_match(boundaries, FUZZY_THRESHOLD)

Skip the cell if no overrides are needed.


In [8]:
# 6. Override scratchpad. Edit and re-run as needed.
# boundaries.at[420, "observed_first_text"] = "(L)(Dec.1779) ... 750.00"
# boundaries = compute_match(boundaries, FLAG_THRESHOLD)

boundaries[boundaries["needs_fix"]][view_cols]


,claimed_first_text,observed_first_text,first_ratio,claimed_last_text,observed_last_text,last_ratio,needs_fix
page,,,,,,,
428,"PORT ROYAL/Va.(1838-54;30;PAID,FREE;Red,Black) ... 20.00","PORT ROYAL/Va.(1838-54;30;PAID,FREE;Red,Black) ... 20.00",1.000000,"Same(large letters,""D""&""A""high,dash at bottom)(1825-33;32;FREE,PAID;Red) ................... 30.00",Same(1833;32;STEAMBOAT;Black) See Inland Waterways Listing . . . . . . . . . . . . . . . . . . . ---,0.290323,True
431,"WOODSTOCK/Va.(1840-50's;31;PAID,5,10,3;Red,Black) . 20.00","WOODSTOCK/Va.(1840-50's;31;PAID,5,10,3;Red,Black) . 20.00",1.000000,Bellefair Mills . . . . . . . . . . 1854 ..... 100.00,Cabin Point . . . . . . . . . . . . 1856 . . . . . . . . . . . . . . . . . . 20.00,0.590909,True
432,Bells Valley . 1851 15.00,(1)Cacapon Depot . . . . . . . 1852 . . . . . . . . . . . . . . . 35.00,0.434783,Calvin's Station ... 1853 . 15.00,)Green Bank . . . . . . . . . . . . 1842 . . . . . . . . . . . . . . . . 40.00,0.409091,True
433,Concord Academy 1843 . 20.00,(1)Green Bottom . . . . . . . . . . 1829-31 . . . . . . . . . . . . . . . . 60.0,0.423077,Independence Hill 1855 . . . . . . . . . . . .. 20.00,*Milton . . . . . . . . . . . . . . . . . . 1811-22 . . . . . . . . . . . . . . . ---,0.315789,True
434,Jackson 1846-51 . 15.00,Mint Spring . . . . . . . . . . . . 1850 . . . . . . . . . . . . . . . . . 20.00,0.476190,(1)Nicholas C.H 1843-47 . 40.00,)Saint Mary's . . . . . . . . . . 1853-54 . . . . . . . . . . . . . . . 40.00,0.592593,True
435,Norfolk .............. . 1825 . 40.00,or Salem . . . . . . . . . . . . . 1819-44 . . . . . . . . . . . . . . 40.00,0.666667,York 1798 . ............. 100.00,"*York Town . . . . . . . . . . . . . 1810,1831 . . . . . . . . . . . . . . . ---",0.562500,True


## 7. Apply page-slide corrections (anchor walk)

The CSV row order is correct -- only the `Page` column is wrong. We use BOTH
the vision-extracted first AND last entry of each page as anchors, walking
through the CSV in row order with a forward-only cursor. This gives two
anchor points per page, makes large-block mis-attributions tractable, and
removes the all-or-nothing failure mode of the previous algorithm.

For each page in ascending order:

1. Find `observed_first_text` in the CSV starting from `cursor` (initially 0).
   Accept if normalized score >= `SLIDE_THRESHOLD`.
2. Find `observed_last_text` in the CSV starting from the accepted
   `first_idx`. Accept if score >= `SLIDE_THRESHOLD`.
3. Advance cursor past `last_idx` (or `first_idx` if last not accepted).

After the walk, redistribute the `Page` column: rows from each accepted page's
`first_idx` through its `last_idx` (or up to the next accepted page's
`first_idx`, if last was not anchored) are reassigned to that page. Rows in
unresolved gaps keep their original Page label.

Status values per page: `ok`, `slid +N`, `slid -N`, `unresolved_first`, or
`slid +N (last unresolved)`.


In [9]:
# 7. Anchor walk.

def best_match_in_range(df: pd.DataFrame, target: str, start: int, end: int):
    """Linear scan: return (best_row_idx, best_score) for rows in [start, end)
    whose Listing best matches target by ratio() (normalized + asymmetric).
    (None, 0.0) if range is empty or target is empty."""
    if target is None or start >= end:
        return (None, 0.0)
    if not normalize(target):
        return (None, 0.0)
    best_idx, best_score = None, 0.0
    for idx in range(start, end):
        s = ratio(target, df.at[idx, "Listing"])
        if s > best_score:
            best_idx, best_score = idx, s
    return (best_idx, best_score)


def anchor_walk(df: pd.DataFrame, boundaries: pd.DataFrame, slide_threshold: float):
    pages = sorted(boundaries.index.tolist())
    n = len(df)
    cursor = 0
    accepted = []   # list of (page, first_idx, last_idx_or_None)
    diag_rows = []

    for page in pages:
        first_text = boundaries.at[page, "observed_first_text"]
        last_text  = boundaries.at[page, "observed_last_text"]

        first_idx, first_score = best_match_in_range(df, first_text, cursor, n)
        if first_idx is None or first_score < slide_threshold:
            diag_rows.append({
                "page": page,
                "first_idx": first_idx,
                "first_score": round(first_score, 3),
                "last_idx": None,
                "last_score": 0.0,
                "_status": "unresolved_first",
            })
            continue

        last_idx, last_score = best_match_in_range(df, last_text, first_idx, n)
        if last_idx is None or last_score < slide_threshold:
            accepted.append((page, first_idx, None))
            diag_rows.append({
                "page": page,
                "first_idx": first_idx,
                "first_score": round(first_score, 3),
                "last_idx": None,
                "last_score": round(last_score, 3),
                "_status": "anchored_first_only",
            })
            cursor = first_idx + 1
            continue

        accepted.append((page, first_idx, last_idx))
        diag_rows.append({
            "page": page,
            "first_idx": first_idx,
            "first_score": round(first_score, 3),
            "last_idx": last_idx,
            "last_score": round(last_score, 3),
            "_status": "anchored",
        })
        cursor = last_idx + 1

    # Redistribute Page column for accepted pages only. Unresolved pages and
    # rows in the gap between an accepted last_idx and the next first_idx
    # retain their original Page label.
    df_corrected = df.copy()
    for i, (page, first_idx, last_idx) in enumerate(accepted):
        if last_idx is not None:
            end_idx = last_idx
        elif i + 1 < len(accepted):
            end_idx = accepted[i + 1][1] - 1
        else:
            end_idx = n - 1
        df_corrected.loc[first_idx:end_idx, "Page"] = page

    diag = pd.DataFrame(diag_rows).set_index("page")
    diag["csv_first"] = boundaries["claimed_first_idx"]
    diag["csv_last"]  = boundaries["claimed_last_idx"]

    def label(row):
        if row["_status"] == "unresolved_first":
            return "unresolved_first"
        shift = int(row["first_idx"] - row["csv_first"])
        sign = "+" if shift > 0 else ""
        base = "ok" if shift == 0 else f"slid {sign}{shift}"
        if row["_status"] == "anchored_first_only":
            base += " (last unresolved)"
        return base

    diag["status"] = diag.apply(label, axis=1)
    diag = diag[["csv_first", "csv_last", "first_idx", "first_score",
                 "last_idx", "last_score", "status"]]

    unresolved = [int(p) for p in diag.index if diag.at[p, "status"].startswith("unresolved")]
    return df_corrected, diag, unresolved


df_corrected, diagnostic, unresolved = anchor_walk(df, boundaries, SLIDE_THRESHOLD)
print(f"unresolved pages: {unresolved}")
diagnostic


unresolved pages: []


,csv_first,csv_last,first_idx,first_score,last_idx,last_score,status
page,,,,,,,
419,0,35,0,1.000,37.0,1.000,ok
420,36,80,38,1.000,80.0,1.000,slid +2
421,81,122,81,1.000,122.0,1.000,ok
422,123,182,123,1.000,182.0,1.000,ok
423,183,249,183,1.000,249.0,1.000,ok
424,250,317,250,1.000,317.0,1.000,ok
425,318,377,318,1.000,377.0,1.000,ok
426,378,433,378,1.000,433.0,1.000,ok
427,434,501,434,0.976,501.0,1.000,ok


In [10]:
# 7b. Migration matrix: which old-page rows ended up on which new-page.
moved_mask = df_corrected["Page"] != df["Page"]
moved = df_corrected[moved_mask]
print(f"rows repaged: {len(moved):,} of {len(df):,}")
if len(moved):
    print()
    print("old -> new page migration matrix:")
    print(pd.crosstab(
        df.loc[moved.index, "Page"],
        df_corrected.loc[moved.index, "Page"],
        rownames=["old"], colnames=["new"],
    ))


rows repaged: 384 of 1,442

old -> new page migration matrix:
new  419  431  432  433  434
old                         
420    2    0    0    0    0
432    0   68    0    0    0
433    0    0  105    0    0
434    0    0    0  109    0
435    0    0    0    0  100


In [11]:
# 8. Write corrected CSV.
out = df_corrected.drop(columns=["_orig_row"])
out.to_csv(OUTPUT_CSV, index=False)
print(f"wrote: {OUTPUT_CSV}")
print(f"rows:  {len(out):,}")
print(f"pages: {out.Page.min()}-{out.Page.max()}")
print()
print(out.groupby("Page").size().rename("rows").to_frame())


wrote: wip/out/VA_ASCC_CTLG.csv
rows:  1,442
pages: 419-435

      rows
Page      
419     38
420     43
421     42
422     60
423     67
424     68
425     60
426     56
427     68
428     47
429     83
430     76
431    116
432    156
433    164
434    150
435    148


## 9. Verification

* First run hits the API for 17 pages x 2 halves = 34 calls. Subsequent runs
  read `wip/cache/<BASENAME>_vision.json` and make zero API calls.
* Eyeball stage 5 for any vision misreads. If found, use the override cell in
  stage 6 and re-run downstream.
* Inspect the section-7 `diagnostic` frame. Every page should end with a
  `status` of `ok`, `slid +N`, or `slid -N`. Any `unresolved_first` or
  `(last unresolved)` row needs manual attention via the section-6 override.
* Spot-check: pick one row from the migration matrix where `old != new`,
  open the matching PNG, confirm the entry visually belongs on `new`.
* Confirm row count of output equals input (no rows added or dropped); only
  the `Page` column should differ.


## 10. Verify Images Above counts (for non-zero claims)

The OCR's image-detection (zero vs non-zero images above each entry) is
trusted -- only the count is suspect when it claims at least one image.
Two known failure modes for the count:

1. **Over-count**: a single stamp with a small adjacent annotation (date,
   minute marking) gets read as two distinct images.
2. **Under-count**: two genuinely separate stamps placed close together
   get merged into one.

This stage hands the model the page scans plus only the listings the OCR
claims have `Images Above > 0`, asks for an array of counts in the same
order, and auto-corrects rows where the model disagrees. Rows where the
model returns null (couldn't locate the entry on the scan) keep their
CSV value and are surfaced for manual review.

Requires `df_corrected` from section 7 to be in memory. Section 14
overwrites the intermediate CSV that section 8 wrote.


In [12]:
# 11. Vision call for image-count verification (one half-page per call).
#
# Scope: we ONLY verify rows where the CSV already claims Images Above > 0.
# Failure mode being addressed by this design: when -L and -R were sent
# together the model mis-attributed images across the L/R seam. We now make
# TWO calls per page, one for each half, sending the full listing list to
# both. The model returns null for any entry it cannot locate on the half
# it was given. The caller merges results: an entry should show up non-null
# on exactly one half.

IMAGES_VISION_MODEL      = "claude-sonnet-4-6"
IMAGES_PROMPT_VERSION    = "v8"   # v7 sent both halves together; v8 splits into per-half calls
IMAGES_MAX_TOKENS        = 4096
IMAGES_CACHE_FILE        = Path(f"./wip/cache/{BASENAME}_images_vision.json")
IMAGES_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

IMAGES_SYSTEM_PROMPT = """You are reading scans of an old American Stampless Cover (ASCC) catalog.

You will receive ONE PNG image per request: a HALF of a catalog page.
The user will tell you whether it is the TOP half (-L) or the BOTTOM
half (-R). The page is a column of catalog entries read top-to-bottom;
your half-page contains only the entries that appear in that half.

The user will hand you a numbered list of catalog entries that the
upstream OCR believes have at least one stamp image above them on the
full page. SOME of these entries appear on your half-page; the others
appear on the OTHER half (which you cannot see) and you should return
null for them. For each entry that DOES appear on your half-page, count
the number of distinct stamp images that appear in the vertical gap
directly ABOVE that entry's text on the scan, between the previous
catalog entry's text on the same half and this entry's text. For an
entry that is the topmost entry on your half-page, count the images
between the column-top / running head / section header (or the very top
of the half-page if your half is -R) and that entry's text.

ONE IMAGE = ONE INDEPENDENT IMPRESSION

The unit you are counting is the INDEPENDENT IMPRESSION: one strike of
a single handstamp or one rendering of a single postmark. A single
handstamp can contain multiple text or numeric elements struck together
as one design -- those multiple elements still count as ONE image
because they came from one stamp.

KINDS OF GRAPHIC ELEMENTS YOU WILL SEE

  1) POSTMARK DEPICTION (one image)
     Spells out the PLACE NAME, as a cursive handwritten script sample
     or a stylized printed postmark in display type (often all-caps,
     sometimes inside an oval/circular frame, sometimes with the date
     integrated into the design).
     Examples: cursive "Burg", printed "FREDS BURG", an oval
     "FINCASTLE Va." postmark, a circular "MOSSY CREEK / AUGUSTA Co. /
     Va. / 1859" cancellation.

  2) RATE MARKING (one image)
     A small standalone handstamp containing just a NUMERAL (sometimes
     within a frame) that indicates a postage rate -- "5", "10", "3",
     "26". Independently struck, counts as a SEPARATE image.

  3) AUXILIARY MARKING (one image)
     A standalone handstamp with a short word like "PAID", "FREE",
     "DUE", "STEAMBOAT", "FORWARDED", "FREE" inside a frame.
     Independently struck, counts as a SEPARATE image.

  4) COMBINED HANDSTAMP (still ONE image)
     One impression that integrates a word marking AND a numeral
     together -- e.g. "PAID 3" struck as one stamp where PAID sits
     atop a "3" (or PAID followed by 3 inside one shared frame),
     "STEAMBOAT 5", "PAID 5". The whole composite is ONE physical
     handstamp and counts as ONE image even though you can read both
     a word and a number in it.

  5) DATE ANNOTATION (does NOT count as its own image)
     Text whose content is a CALENDAR DATE: a month name or
     abbreviation plus a day, optionally a year. Examples:
     "March 7.", "OCT 9", "AUG 1", "Oct.24", "July 2", "FEBR 4".
     A date annotation belongs to the postmark depiction it sits next
     to and is NOT counted separately. The date you see in the
     annotation will match the date listed in the entry's
     parenthetical metadata.

USING THE METADATA NOTATION TO DISAMBIGUATE

The catalog entry's parenthetical metadata is your cue for the markings
field. Inside the markings field, the punctuation signals separate vs
combined:

  * COMMA  ","  -> separate, independently struck handstamps
                   "5,PAID,FREE" means three SEPARATE markings.
  * SLASH  "/"  -> combined into ONE handstamp impression
                   "PAID/3"     means ONE stamp with PAID and 3 together.
                   "PAID/FREE"  means ONE combined PAID-and-FREE stamp.

So if the metadata field is "FREE,PAID/3[F]" the catalog is naming TWO
markings: a standalone "FREE" handstamp, and a combined "PAID 3" hand-
stamp. (Brackets like [F] denote a sub-style and do not change the
count.) If you see a "PAID 3" composite above an entry whose metadata
contains "PAID/3", count it as ONE image, not two.

DECISION RULE FOR ANY ELEMENT YOU SEE

When you see a graphic element above an entry, classify it:

  * Spells the PLACE NAME (script or display type)?
      -> postmark depiction. One image.
  * A bare numeral standing alone (not part of a word-marking design)?
      -> rate marking. One image.
  * A standalone word like PAID / FREE / DUE / STEAMBOAT?
      -> auxiliary marking. One image.
  * A word marking and a numeral struck TOGETHER as one impression
    (shared frame, shared baseline, visibly one stamp)?
      -> combined handstamp. ONE image, not two.
  * Text containing a MONTH NAME plus a day, sitting next to a postmark?
      -> date annotation. Do NOT count it.

EXAMPLES (correct counts) drawn from this catalog:

ONE-image cases:

  Postmark "ALEX" + date "AUG 1"           -> 1
  Postmark "FREDS BURG" + date "March 7."  -> 1
  Postmark "RICHD." + date "Oct.24"        -> 1

TWO-image cases:

  Cursive "Wbg" + cursive "Burg"               -> 2
    (two postmark depictions of "Williamsburg" -- (E)/(L) pattern)
  Postmark "FINCASTLE Va." + bare "5"          -> 2
    (postmark + rate marking; metadata: "30;5,PAID,FREE")
  Postmark "MOSSY CREEK..." + combined "PAID 3" -> 2
    (postmark + ONE combined handstamp; metadata: "37;FREE,PAID/3[F]")
  Postmark + standalone "PAID"                 -> 2

THREE-image cases:

  Postmark + bare "5" + standalone "PAID"      -> 3
    (one postmark, two separate handstamps)

WHAT DOES NOT COUNT AS A STAMP IMAGE AT ALL

  * section headers in plain printed type (state name, region name,
    plain town header) without postmark styling -- typesetting
  * the column header row "Town  Postmark  Dates Seen  Size  Color  Value"
  * page numbers and running heads (e.g. "VIRGINIA" at the top)
  * dotted leader lines, prices, and other body-text artifacts

OUTPUT FORMAT

Return STRICT JSON only -- no surrounding prose, no markdown fences:

  {"counts": [N0, N1, ..., N_{k-1}]}

The array MUST have exactly as many elements as the numbered listings
the user gave you, in the same order. Each element is:

  * A non-negative integer -- the count for an entry visible on this
    half-page. Typically 1, 2, or 3.
  * null -- for an entry NOT visible on this half-page (it appears on
    the other half), or one you cannot locate.
"""


def _parse_counts_response(text, n_expected, stop_reason):
    """Parse {"counts":[...]} tolerantly. Returns the counts list of length
    n_expected. Raises ValueError on unrecoverable parse problems with a
    snippet of the raw text and stop_reason for debugging."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    obj = None
    try:
        obj = json.loads(cleaned)
    except json.JSONDecodeError:
        lo, hi = cleaned.find("{"), cleaned.rfind("}")
        if lo != -1 and hi > lo:
            try:
                obj = json.loads(cleaned[lo:hi + 1])
            except json.JSONDecodeError:
                pass
    if obj is None:
        snippet = cleaned[:500].replace("\n", "\\n")
        raise ValueError(
            f"could not parse JSON (stop_reason={stop_reason!r}, "
            f"text_len={len(text)}): {snippet!r}"
        )
    counts = obj.get("counts")
    if not isinstance(counts, list):
        raise ValueError(f"'counts' missing or not a list: {obj!r}")
    if len(counts) != n_expected:
        raise ValueError(f"counts length {len(counts)} != expected {n_expected}")
    return counts


def extract_half_image_counts(page, half, listings):
    """Send ONE half-page PNG (-L or -R) plus the full listing set to verify;
    return list of (int | None) of length len(listings). The model returns
    null for any listing that does not appear on this half-page."""
    assert half in ("L", "R")
    img_path = IMAGES_DIR / f"page-{page:04d}-{half}.png"
    half_label = "TOP half (-L)" if half == "L" else "BOTTOM half (-R)"
    n = len(listings)
    listings_block = "\n".join(f"  {i}: {s}" for i, s in enumerate(listings))
    open_brace = "{"
    user_prompt = (
        f"This image is the {half_label} of one ASCC catalog page. The "
        f"OCR believes the following {n} catalog entries on the FULL page "
        f"have at least one stamp image above them. SOME of these entries "
        f"appear on the half-page you can see; the others appear on the "
        f"other half. They are listed in reading order top-to-bottom "
        f"across the full page:\n\n"
        f"{listings_block}\n\n"
        f"For each listed entry, return strict JSON of the form "
        f'{{"counts": [N0, N1, ..., N{n-1}]}}. The array MUST have exactly '
        f"{n} elements in the same order as the listings above. For each "
        f"entry visible on this half-page, return a non-negative integer "
        f"count of stamp images directly above it. For each entry NOT "
        f"visible on this half-page (it lives on the other half), return "
        f"null. Reminders: count INDEPENDENT IMPRESSIONS (one strike of "
        f"one stamp = one image). A combined handstamp like 'PAID 3' "
        f"struck as a single design is ONE image, not two; the metadata "
        f"uses '/' to mark such combinations and ',' to mark separately "
        f"struck handstamps. A postmark with a date annotation (month + "
        f"day) next to it is ONE image. A postmark with a bare numeral "
        f"rate marking ('5', '10', '3') is TWO images. Begin your "
        f"response with the opening {open_brace} -- no preamble."
    )
    msg = client.messages.create(
        model=IMAGES_VISION_MODEL,
        max_tokens=IMAGES_MAX_TOKENS,
        system=[{
            "type": "text",
            "text": IMAGES_SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": base64.standard_b64encode(img_path.read_bytes()).decode(),
            }},
            {"type": "text", "text": user_prompt},
        ]}],
    )
    text_blocks = [b.text for b in msg.content if b.type == "text"]
    text = text_blocks[0] if text_blocks else ""
    return _parse_counts_response(text, n, getattr(msg, "stop_reason", "unknown"))


def merge_half_counts(counts_l, counts_r):
    """Combine -L and -R per-listing counts. For each index, prefer the
    non-null value. If both are non-null, take L and record a conflict
    (returned as a list of (index, l, r) tuples). If both null, the entry
    stays null."""
    assert len(counts_l) == len(counts_r)
    merged = []
    conflicts = []
    for i, (l, r) in enumerate(zip(counts_l, counts_r)):
        if l is None and r is None:
            merged.append(None)
        elif l is None:
            merged.append(r)
        elif r is None:
            merged.append(l)
        else:
            merged.append(l)
            conflicts.append((i, l, r))
    return merged, conflicts

print(f"images vision model: {IMAGES_VISION_MODEL}  (prompt {IMAGES_PROMPT_VERSION})")


images vision model: claude-sonnet-4-6  (prompt v8)


In [13]:
# 12. Cache + loop. Two API calls per page (one per half) for pages that
# have at least one CSV row with Images Above > 0. Pages with all zeros
# are skipped. Per-half counts are cached separately and merged in code.

def load_images_cache(path):
    if not path.exists():
        return {"model": IMAGES_VISION_MODEL, "prompt_version": IMAGES_PROMPT_VERSION, "responses": {}}
    cache = json.loads(path.read_text())
    if cache.get("model") != IMAGES_VISION_MODEL or cache.get("prompt_version") != IMAGES_PROMPT_VERSION:
        print(f"images cache invalidated (was model={cache.get('model')!r}, prompt={cache.get('prompt_version')!r})")
        return {"model": IMAGES_VISION_MODEL, "prompt_version": IMAGES_PROMPT_VERSION, "responses": {}}
    return cache

def save_images_cache(path, cache):
    path.write_text(json.dumps(cache, indent=2))

images_cache = load_images_cache(IMAGES_CACHE_FILE)
images_responses = images_cache["responses"]

# Per page: select the rows with Images Above > 0, in CSV row order.
to_verify = (
    df_corrected[df_corrected["Images Above"] > 0]
    .sort_index()
    [["_orig_row", "Page", "Listing", "Images Above"]]
    .copy()
)
print(f"rows to verify: {len(to_verify):,}")
print(to_verify.groupby("Page").size().rename("rows_to_verify"))
print()

calls_made = 0
all_conflicts = []   # list of (page, index, listing, l_count, r_count)

for page, sub in to_verify.groupby("Page", sort=True):
    listings = sub["Listing"].tolist()
    orig_rows = sub["_orig_row"].tolist()
    n = len(listings)
    counts_by_half = {}
    for half in ("L", "R"):
        key = f"page-{int(page):04d}-images-{half}"
        if key in images_responses and not FORCE_REFRESH:
            cached_n = len(images_responses[key].get("counts", []))
            if cached_n != n:
                print(f"  {key}: cached length {cached_n} != current rows {n}; refetching")
            else:
                counts_by_half[half] = images_responses[key]["counts"]
                continue
        try:
            counts = extract_half_image_counts(int(page), half, listings)
        except Exception as exc:
            print(f"  {key}: error {exc!r}")
            continue
        images_responses[key] = {"orig_rows": orig_rows, "counts": counts}
        save_images_cache(IMAGES_CACHE_FILE, images_cache)
        calls_made += 1
        non_null = sum(1 for c in counts if c is not None)
        print(f"  {key}: {non_null}/{n} non-null")
        counts_by_half[half] = counts

    # Merge halves and record any conflicts.
    if "L" in counts_by_half and "R" in counts_by_half:
        merged, conflicts = merge_half_counts(counts_by_half["L"], counts_by_half["R"])
        for i, l, r in conflicts:
            all_conflicts.append((int(page), int(orig_rows[i]), listings[i], l, r))
        merged_key = f"page-{int(page):04d}-images"
        images_responses[merged_key] = {"orig_rows": orig_rows, "counts": merged}
        save_images_cache(IMAGES_CACHE_FILE, images_cache)

print(f"\napi calls this run: {calls_made};  half-keys cached: {sum(1 for k in images_responses if k.endswith(('-L','-R')))}")

if all_conflicts:
    print()
    print(f"L/R conflicts ({len(all_conflicts)}) -- both halves returned a non-null count; took L:")
    for page, orig, lst, l, r in all_conflicts:
        print(f"  page {page} row {orig}: L={l} R={r} | {lst[:80]}")

# Build observed_df keyed by _orig_row from the merged per-page responses.
observed_rows = []
for key, payload in images_responses.items():
    if not key.endswith("-images"):
        continue
    page = int(key.split("-")[1])
    counts    = payload.get("counts", [])
    orig_rows = payload.get("orig_rows", [])
    if len(counts) != len(orig_rows):
        print(f"  warning: {key}: counts/orig_rows length mismatch ({len(counts)} vs {len(orig_rows)})")
        continue
    for orig, c in zip(orig_rows, counts):
        observed_rows.append({
            "_orig_row":             int(orig),
            "page":                  page,
            "images_above_observed": (None if c is None else int(c)),
        })
observed_df = pd.DataFrame(observed_rows)
print(f"\nobserved entries total: {len(observed_df):,}")
observed_df.head(8)


images cache invalidated (was model='claude-opus-4-7', prompt='v7')
rows to verify: 159
Page
419    18
420    26
421    11
422    15
423    13
424     8
425    12
426    12
427     8
428    17
429     7
430     8
431     4
Name: rows_to_verify, dtype: int64

  page-0419-images-L: 7/18 non-null
  page-0419-images-R: 11/18 non-null
  page-0420-images-L: 12/26 non-null
  page-0420-images-R: 14/26 non-null
  page-0421-images-L: 5/11 non-null
  page-0421-images-R: 6/11 non-null
  page-0422-images-L: 6/15 non-null
  page-0422-images-R: 9/15 non-null
  page-0423-images-L: 7/13 non-null
  page-0423-images-R: 6/13 non-null
  page-0424-images-L: 4/8 non-null
  page-0424-images-R: 4/8 non-null
  page-0425-images-L: 7/12 non-null
  page-0425-images-R: 5/12 non-null
  page-0426-images-L: 6/12 non-null
  page-0426-images-R: 6/12 non-null
  page-0427-images-L: 3/8 non-null
  page-0427-images-R: 5/8 non-null
  page-0428-images-L: 8/17 non-null
  page-0428-images-R: 9/17 non-null
  page-0429-images-L: 

,_orig_row,page,images_above_observed
0,0,419,1
1,3,419,1
2,6,419,1
3,8,419,1
4,10,419,1
5,11,419,1
6,12,419,1
7,14,419,1


In [14]:
# 13. Flag deltas and apply Images Above corrections.
#
# observed_df is keyed by _orig_row and only covers rows the OCR claims have
# Images Above > 0. Anything else keeps its CSV value untouched.

merged = df_corrected.merge(
    observed_df[["_orig_row", "images_above_observed"]],
    on="_orig_row",
    how="left",
).rename(columns={"Images Above": "images_above_csv"})

# Three buckets among the verified rows (those where observed is not missing
# from observed_df at all -- i.e. originally images_above_csv > 0):
#   * unlocated: model returned null
#   * agreement: observed == csv
#   * delta:     observed != csv
verified_mask = merged["_orig_row"].isin(observed_df["_orig_row"])
unlocated_mask = verified_mask & merged["images_above_observed"].isna()
have_obs       = verified_mask & ~merged["images_above_observed"].isna()
delta_mask     = have_obs & (merged["images_above_observed"].astype("Int64") != merged["images_above_csv"].astype("Int64"))

print(f"verified rows:    {verified_mask.sum():,}")
print(f"  agreement:      {(have_obs & ~delta_mask).sum():,}")
print(f"  delta != 0:     {delta_mask.sum():,}")
print(f"  unlocated:      {unlocated_mask.sum():,}")

# Apply corrections: where the model gave a non-null count that disagrees,
# overwrite Images Above. Unlocated rows keep the CSV value.
df_final = df_corrected.copy()
to_apply = merged.loc[delta_mask, ["_orig_row", "images_above_observed"]]
for _, r in to_apply.iterrows():
    df_final.loc[df_final["_orig_row"] == int(r["_orig_row"]), "Images Above"] = int(r["images_above_observed"])
print(f"\nImages Above corrections applied: {len(to_apply):,}")

# Flagged table for visual review.
flagged = merged.loc[delta_mask, ["_orig_row", "Page", "Listing", "images_above_csv", "images_above_observed"]].copy()
flagged["delta"] = flagged["images_above_observed"].astype(int) - flagged["images_above_csv"].astype(int)
if len(flagged):
    print()
    print("delta distribution:")
    print(flagged["delta"].value_counts().sort_index())

# Show unlocated rows separately so they don't get lost in the noise.
unlocated = merged.loc[unlocated_mask, ["_orig_row", "Page", "Listing", "images_above_csv"]]
if len(unlocated):
    print()
    print(f"unlocated rows ({len(unlocated)}) -- model could not find these on the scan:")
    print(unlocated.to_string(index=False))

def highlight_delta(row):
    return ["background-color: #ffe0e0"] * len(row)

flagged.style.apply(highlight_delta, axis=1)


verified rows:    159
  agreement:      121
  delta != 0:     38
  unlocated:      0

Images Above corrections applied: 38

delta distribution:
delta
-2     1
-1    19
 1    15
 2     3
Name: count, dtype: int64


,_orig_row,Page,Listing,images_above_csv,images_above_observed,delta
16,16,419,"NORFOLK(backstamp)(E)(Feb.11,1775;SL-29x5,MOD below;Black) ............. 1500.00",1,0.000000,-1
22,22,419,"SUFFOLK(April 12,1775;SL-30x5,MDD below;Red) .... 1500.00",1,0.000000,-1
24,24,419,"WBurg(Williamsburg)(E)(Nov.14,1734;Ms,Black) .... 2500.00",1,2.000000,1
69,69,420,"Richmond(Aug.23,1782;Ms;Black) . . ......... 600.00",2,1.000000,-1
88,88,421,ABINGDON/VA(1834-36;irregular C-26;Black) 150.00,1,2.000000,1
95,95,421,"AETNA,HANOVER,VA/5(""5""in center,MD at bottom)(1849;27;Black) 300.00",2,1.000000,-1
98,98,421,"ALEXAN(1790-91;SL-19x3,MDD;Black) . . . . .... 200.00",2,1.000000,-1
118,118,421,"BARBOURS.VILLE VA(""b""ms)(1836;SL-39x3;Black-brown) 250.00",2,1.000000,-1
130,130,422,BIG LICK/Va.(1846-47;DC-30;5[neg.or sawtooth C];Black) 125.00,1,3.000000,2
151,151,422,CABIN POINT/VA(1853-56;29;PAID/3[C];Black) 30.00,1,2.000000,1


## 14. Manual override + final write

Override on `df_final` before the write below if you eyeball a flagged
row and decide the model was wrong, or if an unlocated row needs a
specific value:

    df_final.loc[df_final["_orig_row"] == 632, "Images Above"] = 1

Skip the override cell if no edits are needed.


In [15]:
# 14. Override scratchpad + final write.
# df_final.loc[df_final["_orig_row"] == 632, "Images Above"] = 1

out = df_final.drop(columns=["_orig_row"])
out.to_csv(OUTPUT_CSV, index=False)
print(f"wrote: {OUTPUT_CSV}")
print(f"rows:  {len(out):,}")
print(f"pages: {out.Page.min()}-{out.Page.max()}")
print()
print("Images Above value counts (input -> output):")
before = df["Images Above"].value_counts().sort_index().rename("input")
after  = out["Images Above"].value_counts().sort_index().rename("output")
print(pd.concat([before, after], axis=1).fillna(0).astype(int))
print()
print("rows per page:")
print(out.groupby("Page").size().rename("rows").to_frame())


wrote: wip/out/VA_ASCC_CTLG.csv
rows:  1,442
pages: 419-435

Images Above value counts (input -> output):
              input  output
Images Above               
0              1283    1293
1               133     117
2                25      27
3                 1       5

rows per page:
      rows
Page      
419     38
420     43
421     42
422     60
423     67
424     68
425     60
426     56
427     68
428     47
429     83
430     76
431    116
432    156
433    164
434    150
435    148
